In [14]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages   
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated

In [16]:
class ChatState(BaseModel):
    messages:Annotated[list,add_messages]  ## add_messages will append the every new mssg in the list

In [17]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [18]:
## Node
def chatBotNode(state:ChatState) -> ChatState:
    res = llm.invoke(state.messages)
    state.messages = [res]  ## res will atomatically append in the state.messages list by add_messages
    return state

In [19]:
memory = InMemorySaver()

In [20]:
graph = StateGraph(ChatState)
graph.add_node("chatBot",chatBotNode)


## build Edge : we hv build the flow
graph.add_edge(START,"chatBot")
graph.add_edge("chatBot",END)



finalGraph = graph.compile(checkpointer=memory)

In [23]:
res = finalGraph.invoke(
    {"messages" : [{"role":"user", "content":"Hello, what is my name"}]},
    {"configurable":{"thread_id":"1"}}
    )


res

{'messages': [HumanMessage(content='Hello, How are you ? i am nakul', additional_kwargs={}, response_metadata={}, id='e4820f51-273a-4e87-a62e-093c355fc33b'),
  AIMessage(content='Hello Nakul! I’m doing great—thanks for asking. How can I help you today?', additional_kwargs={'reasoning_content': 'User: "Hello, How are you ? i am nakul". They greet. We respond politely, ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 81, 'total_tokens': 137, 'completion_time': 0.077237705, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.004482949, 'prompt_tokens_details': None, 'queue_time': 0.400919062, 'total_time': 0.081720654}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ee07-a37b-7492-abf6-bce05c91992f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'outp

In [24]:
print(res["messages"][-1].content)

You introduced yourself as **Nakul**. If that’s not right, just let me know!
